# `scVI` correction, leiden clustering and manual annotation of the mesenchymal cells

**Developed by:** Anna Maguza\
**Affiliation:** Faculty of Medicine, Würzburg University\
**Creation date:** 31st January 2025\
**Last modified date:** 31st January 2025

#### **Objective**

This notebook continues the process for annotation of mesenchymal cell states. We already annotated cells by label transfer and validated predicted cell states by labels transfer, the purpose of the next steps are:
1. Annnotate subpopulations of stromal cells (S1, S2, S3, S4)
2. Annnotate subpopulations of muscles and myofibroblasts
3. Annnotate subpopulations of mesoderm

## Import packages

In [1]:
import scvi
import anndata
import warnings
import numpy as np
import scanpy as sc
import anndata
import pandas as pd
import plotnine as p
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scib_metrics.benchmark import Benchmarker

import json
from datetime import datetime

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup working environment

In [2]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi = 180, color_map = 'magma_r', dpi_save = 300, vector_friendly = True, format = 'svg')

In [3]:
warnings.simplefilter(action = 'ignore')
scvi.settings.seed = 1712
%config InlineBackend.print_figure_kwargs = {'facecolor' : "w"}
%config InlineBackend.figure_format = 'retina'

Seed set to 1712


In [4]:
arches_params = dict(
    use_layer_norm = "both",
    use_batch_norm = "none",
    encode_covariates = True,
    dropout_rate = 0.2,
    n_layers = 3,
)

In [5]:
def X_is_raw(adata):
    return np.array_equal(adata.X.sum(axis=0).astype(int), adata.X.sum(axis=0))

In [6]:
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

In [232]:
timestamp

'02022025_163814'

## Upload data

In [7]:
adata = sc.read_h5ad('data/gut_data/gut_hs_all_datasets_scVI_scANVI_mesenchymal_cellstates_AM_30012025_144410_raw.h5ad')
adata

AnnData object with n_obs × n_vars = 185027 × 43704
    obs: 'cell_index', 'Source Name', 'ENA_SAMPLE', 'BioSD_SAMPLE', 'organism', 'disease', 'organism_part', 'cell_type', 'growth_condition', 'developmental_stage', 'Material Type', 'Protocol REF', 'sample_id', 'LIBRARY_LAYOUT', 'cdna_read_size', 'cell_barcode_size', 'end_bias', 'library_construction', 'sample_barcode_size', 'umi_barcode_offset', 'umi_barcode_size', 'Performer', 'Assay Name', 'ENA_EXPERIMENT', 'ENA_RUN', 'time', 'time_unit', 'n_genes', 'doublet_scores', 'predicted_doublets', 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'percent_mito', 'n_counts_mito', 'percent_ribo', 'n_counts_ribo', 'percent_hb', 'n_counts_hb', 'percent_top50', 'cell_passed_qc', 'qc_cluster', 'cluster_passed_qc', 'consensus_fraction', 'consensus_passed_qc', 'total_counts', 'n_genes_by_counts', 'percent_chrY', 'XIST-counts', 'XIST-percentage', 'sex', 'S_score', 'G2M_score', 'Cell_cycle_phase', 'Study_name', 'ArrayExpress_ID', 'library_preparation_pro

In [8]:
adata.obs.index = adata.obs.index.astype(str)

In [9]:
adata.obs_names_make_unique()

In [10]:
duplicated_obs = adata.obs.index.duplicated().sum()
print(f"Number of duplicated observation indexes: {duplicated_obs}")

Number of duplicated observation indexes: 0


In [11]:
region_map = {
    'duodenum': 'small intestine',
    'ileum': 'small intestine',
    'small intestine': 'small intestine',
    'terminal ileum': 'small intestine',
    'jejunum': 'small intestine',
    'sigmoid colon': 'large intestine',
    'caecum': 'large intestine',
    'ascending colon': 'large intestine',
    'transverse colon': 'large intestine',
    'large intestine': 'large intestine',
    'descending colon': 'large intestine',
    'appendix': 'large intestine',
    'colon': 'large intestine',
    'rectum': 'large intestine'
}

adata.obs['gut_region'] = adata.obs['organism_part'].map(region_map)

In [12]:
adata.obs['cell_states'] = adata.obs['cellstates_scANVI'].copy()

## Mesoderm

In [13]:
adata.obs['cellstates_scANVI'].value_counts()

cellstates_scANVI
Mesoderm           70494
Stromal            44200
SMC                22427
Myofibroblast      19010
Pericyte           15702
Mesothelium        13159
ICC                   31
Lymphoid_Stroma        4
Name: count, dtype: int64

In [14]:
adata_mesoderm = adata[adata.obs['cellstates_scANVI'] == 'Mesoderm']

### Extract highly variable genes

In [15]:
adata_mesoderm.layers['counts'] = adata_mesoderm.X.copy()

In [16]:
sc.pp.highly_variable_genes(
    adata_mesoderm,
    flavor = "seurat_v3",
    n_top_genes = 1500,
    layer = "counts",
    batch_key = "library_preparation_protocol",
    subset = True,
    span = 1
)

extracting highly variable genes


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


--> added
    'highly_variable', boolean vector (adata.var)
    'highly_variable_rank', float vector (adata.var)
    'means', float vector (adata.var)
    'variances', float vector (adata.var)
    'variances_norm', float vector (adata.var)


## Run scVI

In [17]:
scvi.model.SCVI.setup_anndata(adata_mesoderm, 
                              categorical_covariate_keys=['sample_id', 'library_construnction_and_layout', 'Protocol REF'],
                              layer = 'counts')

In [18]:
scvi_model = scvi.model.SCVI(adata_mesoderm,
                            n_latent = 10, 
                            n_hidden = 128,
                            n_layers = 2, 
                            dropout_rate = 0.1,
                            dispersion = 'gene-batch', 
                            gene_likelihood = 'nb')

In [19]:
scvi_model.train(100, 
                early_stopping = True,
                early_stopping_patience = 10,
                check_val_every_n_epoch = 1, 
                enable_progress_bar = True)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Epoch 100/100: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [09:10<00:00,  5.95s/it, v_num=1, train_loss_step=464, train_loss_epoch=473]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100/100: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [09:10<00:00,  5.51s/it, v_num=1, train_loss_step=464, train_loss_epoch=473]


In [20]:
adata_mesoderm.obsm["X_scVI"] = scvi_model.get_latent_representation(adata_mesoderm)

#### Evaluate model performance using the [_Svensson_](https://www.nxn.se/valent/2023/8/10/training-scvi-posterior-predictive-distributions-over-epochs) method

In [21]:
history_df = (
    scvi_model.history['elbo_train'].astype(float)
    .join(scvi_model.history['elbo_validation'].astype(float))
    .reset_index()
    .melt(id_vars=['epoch'])
)

In [ ]:
plt.figure(figsize=(12, 10))

plt.plot(history_df[history_df['variable'] == 'elbo_train']['epoch'], 
         history_df[history_df['variable'] == 'elbo_train']['value'], 
         color='black', marker='o', label='Training ELBO')

plt.plot(history_df[history_df['variable'] == 'elbo_validation']['epoch'],
         history_df[history_df['variable'] == 'elbo_validation']['value'], 
         color='red', marker='o', label='Validation ELBO')

plt.xlabel('Epoch')
plt.ylabel('ELBO Value')
plt.title('Training and Validation ELBO over Epochs')
plt.legend()
plt.grid(True, alpha=0.3)

plt.xlim(left=1)

plt.tight_layout()
plt.show()

+ Visualize dataset

In [23]:
sc.pp.neighbors(adata_mesoderm, use_rep = "X_scVI", n_neighbors = 50, metric = 'minkowski')
sc.tl.umap(adata_mesoderm, min_dist = 0.3, spread = 4, random_state = 1712)

computing neighbors
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:20)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:40)


In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(10, 10))
    sc.pl.umap(adata_mesoderm,color=["Study_name", "ArrayExpress_ID", 'age_group', 'organism_part', 'developmental_stage', 'library_preparation_protocol', 'library_construnction_and_layout', 'Protocol REF', 'Performer', 'gut_region'], ncols=4, frameon=False, show=False, size = 5)
    plt.savefig(f"figures/mesoderm_categorical_values1_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(10, 10))
    sc.pl.umap(adata_mesoderm,color=['immunophenotype', 'sex', 'Cell_cycle_phase', 'full_age', 'growth_condition', 'sampling_site', 'Material Type', 'donor_id', 'library_construnction_and_layout', 'Protocol REF', 'Performer', 'gut_region'], ncols=4, frameon=False, show=False, size = 5)
    plt.savefig(f"figures/mesoderm_categorical_values2_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_mesoderm,color=['n_genes', 'n_counts', 'total_counts', 'n_genes_by_counts', 'predicted_doublets', 'percent_mito', 'percent_ribo', 'percent_hb'], color_map = 'magma_r', ncols=4, frameon=False, show=False, size = 5)
    plt.savefig(f"mesoderm_continues_values1_{timestamp}.png", bbox_inches="tight")
    plt.show()

## Manual Annotation

In [25]:
adata_log = adata[adata.obs['cellstates_scANVI'] == 'Mesoderm']
adata_log.obsm['X_umap'] = adata_mesoderm.obsm['X_umap'].copy()
sc.pp.normalize_total(adata_log, target_sum=1e6, exclude_highly_expressed=True)
sc.pp.log1p(adata_log)

normalizing counts per cell. The following highly-expressed genes are not considered during normalization factor computation:
['MFSD1', 'ATG10', 'GPX3', 'MLN', 'COL1A2', 'GSN', 'VIM-AS1', 'C10orf55', 'ADIRF-AS1', 'BEST1', 'TALAM1', 'ENSG00000273149', 'FOS', 'COL1A1', 'FTL', 'MT-RNR1', 'MT-RNR2', 'MT-CO1']
    finished (0:00:00)


* Mesoderm 1(HAND1+) vs Mesoderm 2(ZEB2+)

In [26]:
sc.tl.score_genes(adata_log, ['HAND1', 'HAND2', 'PITX2'], score_name='mesoderm1_score')

computing score 'mesoderm1_score'
    finished: added
    'mesoderm1_score', score of gene set (adata.obs).
    150 total control genes are used. (0:00:00)


In [27]:
adata_log.obs['cell_states'] = 'Mesoderm 2(ZEB2+)'
adata_log.obs['cell_states'] = adata_log.obs['cell_states'].astype('category')
adata_log.obs['cell_states'] = adata_log.obs['cell_states'].cat.add_categories(['Mesoderm 1(HAND1+)'])

In [28]:
adata_log.obs.loc[adata_log.obs['mesoderm1_score'] > 2, 'cell_states'] = 'Mesoderm 1(HAND1+)'

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['HAND1', 'HAND2', 'PITX2', 'ZEB2', 'mesoderm1_score', 'cell_states'], ncols=3, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/mesoderm_manual_annotation_mesoderm_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [30]:
all_categories = pd.Categorical(
    pd.concat([adata.obs['cell_states'], adata_log.obs['cell_states']]).unique()
)

adata.obs['cell_states'] = pd.Categorical(
    adata.obs['cell_states'],
    categories=all_categories
)

adata_log.obs['cell_states'] = pd.Categorical(
    adata_log.obs['cell_states'],
    categories=all_categories
)

In [31]:
shared_indices = adata_log.obs.index
adata.obs.loc[shared_indices, 'cell_states'] = adata_log.obs['cell_states']

In [32]:
adata.obs['cell_states'].value_counts()

cell_states
Mesoderm 2(ZEB2+)     68505
Stromal               44200
SMC                   22427
Myofibroblast         19010
Pericyte              15702
Mesothelium           13159
Mesoderm 1(HAND1+)     1989
ICC                      31
Lymphoid_Stroma           4
Mesoderm                  0
Name: count, dtype: int64

In [33]:
del adata_mesoderm, adata_log

## Stromal cells

In [121]:
adata_stromal = adata[adata.obs['cellstates_scANVI'].isin(['Stromal', 'SMC', 'Myofibroblast', 'Lymphoid_Stroma', 'ICC'])]
adata_stromal

View of AnnData object with n_obs × n_vars = 85672 × 43704
    obs: 'cell_index', 'Source Name', 'ENA_SAMPLE', 'BioSD_SAMPLE', 'organism', 'disease', 'organism_part', 'cell_type', 'growth_condition', 'developmental_stage', 'Material Type', 'Protocol REF', 'sample_id', 'LIBRARY_LAYOUT', 'cdna_read_size', 'cell_barcode_size', 'end_bias', 'library_construction', 'sample_barcode_size', 'umi_barcode_offset', 'umi_barcode_size', 'Performer', 'Assay Name', 'ENA_EXPERIMENT', 'ENA_RUN', 'time', 'time_unit', 'n_genes', 'doublet_scores', 'predicted_doublets', 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'percent_mito', 'n_counts_mito', 'percent_ribo', 'n_counts_ribo', 'percent_hb', 'n_counts_hb', 'percent_top50', 'cell_passed_qc', 'qc_cluster', 'cluster_passed_qc', 'consensus_fraction', 'consensus_passed_qc', 'total_counts', 'n_genes_by_counts', 'percent_chrY', 'XIST-counts', 'XIST-percentage', 'sex', 'S_score', 'G2M_score', 'Cell_cycle_phase', 'Study_name', 'ArrayExpress_ID', 'library_preparat

### Extract highly variable genes

In [122]:
adata_stromal.layers['counts'] = adata_stromal.X.copy()

In [123]:
sc.pp.highly_variable_genes(
    adata_stromal,
    flavor = "seurat_v3",
    n_top_genes = 2000,
    layer = "counts",
    batch_key = "library_preparation_protocol",
    subset = True,
    span = 1
)

extracting highly variable genes
--> added
    'highly_variable', boolean vector (adata.var)
    'highly_variable_rank', float vector (adata.var)
    'means', float vector (adata.var)
    'variances', float vector (adata.var)
    'variances_norm', float vector (adata.var)


## Run scVI

In [124]:
scvi.model.SCVI.setup_anndata(adata_stromal, 
                              categorical_covariate_keys=['sample_id', 'library_construnction_and_layout', 'Protocol REF'],
                              layer = 'counts')

In [125]:
scvi_model = scvi.model.SCVI(adata_stromal,
                            n_latent = 50, 
                            n_hidden = 128,
                            n_layers = 2, 
                            dropout_rate = 0.1,
                            dispersion = 'gene-batch', 
                            gene_likelihood = 'nb')

In [126]:
scvi_model.train(100, 
                early_stopping = True,
                early_stopping_patience = 10,
                check_val_every_n_epoch = 1, 
                enable_progress_bar = True)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Epoch 100/100: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [14:39<00:00,  9.29s/it, v_num=1, train_loss_step=539, train_loss_epoch=554]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100/100: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [14:39<00:00,  8.80s/it, v_num=1, train_loss_step=539, train_loss_epoch=554]


In [127]:
adata_stromal.obsm["X_scVI"] = scvi_model.get_latent_representation(adata_stromal)

#### Evaluate model performance using the [_Svensson_](https://www.nxn.se/valent/2023/8/10/training-scvi-posterior-predictive-distributions-over-epochs) method

In [128]:
history_df = (
    scvi_model.history['elbo_train'].astype(float)
    .join(scvi_model.history['elbo_validation'].astype(float))
    .reset_index()
    .melt(id_vars=['epoch'])
)

In [ ]:
plt.figure(figsize=(12, 10))

plt.plot(history_df[history_df['variable'] == 'elbo_train']['epoch'], 
         history_df[history_df['variable'] == 'elbo_train']['value'], 
         color='black', marker='o', label='Training ELBO')

plt.plot(history_df[history_df['variable'] == 'elbo_validation']['epoch'],
         history_df[history_df['variable'] == 'elbo_validation']['value'], 
         color='red', marker='o', label='Validation ELBO')

plt.xlabel('Epoch')
plt.ylabel('ELBO Value')
plt.title('Training and Validation ELBO over Epochs')
plt.legend()
plt.grid(True, alpha=0.3)

plt.xlim(left=1)

plt.tight_layout()
plt.show()

+ Visualize dataset

In [130]:
sc.pp.neighbors(adata_stromal, use_rep = "X_scVI", n_neighbors = 50, metric = 'minkowski')
sc.tl.umap(adata_stromal, min_dist = 0.3, spread = 4, random_state = 1712)

computing neighbors
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:18)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:51)


In [56]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(10, 10))
    sc.pl.umap(adata_stromal,color=["Study_name", "ArrayExpress_ID", 'metadata_cluster', 'age_group', 'organism_part', 'developmental_stage', 'library_preparation_protocol', 'immunophenotype', 'sex', 'Cell_cycle_phase', 'developmental_stage', 'full_age', 'growth_condition', 'sampling_site', 'Material Type', 'donor_id', 'library_construnction_and_layout', 'Protocol REF', 'Performer'], ncols=4, frameon=False, show=False, size = 15)
    plt.savefig(f"figures/Stromal_categorical_values1_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_stromal,color=['n_genes', 'n_counts', 'total_counts', 'n_genes_by_counts', 'predicted_doublets', 'percent_mito', 'percent_ribo', 'percent_hb'], color_map = 'magma_r', ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/Stromal_continues_values1_{timestamp}.png", bbox_inches="tight")
    plt.show()

### `Leiden` clustering

In [131]:
sc.tl.leiden(adata_stromal, resolution = 0.3, random_state = 1786)

running Leiden clustering
    finished: found 11 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:02:18)


In [ ]:
sc.pl.umap(adata_stromal,color=['leiden', 'Integrated_05', 'cellstates_scANVI'], color_map = 'magma_r', ncols=2, frameon=False, show=False, size = 10)

## Manual Annotation

In [84]:
adata_log = adata[adata.obs['cellstates_scANVI'].isin(['Stromal', 'SMC', 'Myofibroblast', 'Lymphoid_Stroma', 'ICC'])]
adata_log.obs['leiden'] = adata_stromal.obs['leiden'].copy()
adata_log.obsm['X_umap'] = adata_stromal.obsm['X_umap'].copy()
sc.pp.normalize_total(adata_log, target_sum=1e6, exclude_highly_expressed=True)
sc.pp.log1p(adata_log)

normalizing counts per cell. The following highly-expressed genes are not considered during normalization factor computation:
['S100A6', 'ACTG2', 'MFSD1', 'ENSG00000286848', 'CXCL10', 'CXCL13', 'ATG10', 'CXCL14', 'LINC01013', 'COL1A2', 'ADAMDEC1', 'CCL19', 'CCL21', 'TPM2', 'VIM-AS1', 'ADIRF-AS1', 'ACTA2', 'BEST1', 'TALAM1', 'TAGLN', 'ENSG00000272173', 'ENSG00000273149', 'THBS1', 'RPLP1', 'HBA2', 'CCL2', 'CCL11', 'COL1A1', 'ENSG00000267598', 'TMSB4X', 'MT-RNR1', 'MT-RNR2', 'MT-CO1', 'MT-CO2']
    finished (0:00:00)


In [ ]:
adata_log.obs['cell_states'] = adata_log.obs['cell_states'].cat.add_categories(
    ['Stromal 1 (ADAMDEC1+)', 'Stromal 2 (NPY+)', 'Stromal 2 (CH25H+)',
     'Stromal 3 (C7+)', 'Transitional Stromal 3 (C3+)', 'SMC (PLPP2+)', 'SMC (PART1/CAPN3+)', 'Myofibroblast (RSPO2+)', 'ICC'])

* Stromal 1 markers

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden','ADAMDEC1', 'ADAM28','CCL11', 'CCL8', 'CCL13', 'FABP5', 'COL6A5', 'IFIT3'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/stromal_stromal1_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [94]:
adata_log.obs.loc[adata_log.obs['leiden'] == '0', 'cell_states'] = 'Stromal 1 (ADAMDEC1+)'
adata_log.obs.loc[adata_log.obs['leiden'] == '7', 'cell_states'] = 'Stromal 1 (ADAMDEC1+)'

* Stromal 2 markers

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden','PDGFRA','BMP4', 'F3', 'NPY', 'CH25H', 'MMP1', 'FRZB', 'BMP5'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/stromal_stromal2_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [95]:
adata_log.obs.loc[adata_log.obs['leiden'] == '2', 'cell_states'] = 'Stromal 2 (NPY+)'

sc.tl.score_genes(adata_log, ['CH25H'], score_name='CH25H_score')
adata_log.obs.loc[(adata_log.obs['CH25H_score'] > 4) & (adata_log.obs['leiden'] == '2'), 'cell_states'] = 'Stromal 2 (CH25H+)'

computing score 'CH25H_score'
    finished: added
    'CH25H_score', score of gene set (adata.obs).
    50 total control genes are used. (0:00:00)


* Stromal 3 markers

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'C7', 'KCNN3', 'LRRC3B'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/stromal_stromal3_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [96]:
adata_log.obs.loc[adata_log.obs['leiden'] == '6', 'cell_states'] = 'Stromal 3 (C7+)'

* Transitional stroma 3 markers

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'C3', 'CXCL13', 'CCL21'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/stromal_transitional_stromal3_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [98]:
adata_log.obs.loc[adata_log.obs['leiden'] == '9', 'cell_states'] = 'Transitional Stromal 3 (C3+)'

* Myofibroblasts

DES shouldnt be expressed!

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'ACTA2', 'TAGLN', 'DCN', 'DES', 'RSPO2'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/stromal_transitional_myofibroblasts_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [99]:
adata_log.obs.loc[adata_log.obs['leiden'] == '1', 'cell_states'] = 'Myofibroblast (RSPO2+)'
adata_log.obs.loc[adata_log.obs['leiden'] == '3', 'cell_states'] = 'Myofibroblast'

* Smooth muscle cells

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'DES', 'CNN1', 'ACTA2', 'PLPP2', 'PART1', 'CAPN3'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/stromal_transitional_smc_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [100]:
adata_log.obs.loc[adata_log.obs['leiden'] == '4', 'cell_states'] = 'SMC (PLPP2+)'
adata_log.obs.loc[adata_log.obs['leiden'] == '5', 'cell_states'] = 'SMC (PART1/CAPN3+)'

* ICC

In [101]:
sc.tl.score_genes(adata_log, ['KIT', 'ANO1', 'ETV1', 'SPON2'], score_name='ICC_score')
adata_log.obs.loc[(adata_log.obs['ICC_score'] > 4) & (adata_log.obs['leiden'] == '5'), 'cell_states'] = 'ICC'

computing score 'ICC_score'
    finished: added
    'ICC_score', score of gene set (adata.obs).
    199 total control genes are used. (0:00:00)


In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'KIT', 'ANO1', 'ETV1', 'SPON2', 'ICC_score'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/stromal_transitional_icc_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

+ Lymphoid stroma (T reticular and Follicular Dendritic Cells (FDCs))

In [102]:
sc.tl.score_genes(adata_log, ['CCL19', 'CCL21', 'LTBR', 'CXCL13', 'CR1', 'CR2', 'FCGR2B', 'MADCAM1'], score_name='Lymphoid_stroma_score')
adata_log.obs.loc[(adata_log.obs['Lymphoid_stroma_score'] > 2) & (adata_log.obs['leiden'] == '7'), 'cell_states'] = 'Lymphoid_Stroma'

computing score 'Lymphoid_stroma_score'
    finished: added
    'Lymphoid_stroma_score', score of gene set (adata.obs).
    300 total control genes are used. (0:00:00)


In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'CCL19', 'CCL21', 'LTBR', 'CXCL13', 'CR1', 'CR2', 'FCGR2B', 'MADCAM1', 'Lymphoid_stroma_score'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/stromal_lymphoid_stroma_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

* Cluster 8

In [103]:
sc.tl.rank_genes_groups(adata_log, groupby="leiden", method="wilcoxon")

ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:02:41)


In [ ]:
df = sc.get.rank_genes_groups_df(adata_log, group="8").head(100)
df['names'].to_list()

In [105]:
adata_log.obs['cell_states'] = adata_log.obs['cell_states'].cat.add_categories(
    ['Cycling stromal'])
adata_log.obs.loc[(adata_log.obs['leiden'] == '8'), 'cell_states'] = 'Cycling stromal'

* Cluster 10

In [ ]:
df = sc.get.rank_genes_groups_df(adata_log, group="10").head(100)
df['names'].to_list()

In [ ]:
adata_log.obs['cell_states'] = adata_log.obs['cell_states'].cat.add_categories(
    ['Mesothelium'])
adata_log.obs.loc[(adata_log.obs['leiden'] == '10'), 'cell_states'] = 'Mesothelium'

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['cellstates_scANVI', 'cell_states'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/stromal_manual_annotation_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [113]:
all_categories = pd.Categorical(
    pd.concat([adata.obs['cell_states'], adata_log.obs['cell_states']]).unique()
)

adata.obs['cell_states'] = pd.Categorical(
    adata.obs['cell_states'],
    categories=all_categories
)

adata_log.obs['cell_states'] = pd.Categorical(
    adata_log.obs['cell_states'],
    categories=all_categories
)

In [114]:
shared_indices = adata_log.obs.index
adata.obs.loc[shared_indices, 'cell_states'] = adata_log.obs['cell_states']

In [116]:
del adata_stromal, adata_log

## Mesothelium

In [133]:
adata_mesothelium = adata[adata.obs['cellstates_scANVI'].isin(['Mesothelium'])]
adata_mesothelium

View of AnnData object with n_obs × n_vars = 13159 × 43704
    obs: 'cell_index', 'Source Name', 'ENA_SAMPLE', 'BioSD_SAMPLE', 'organism', 'disease', 'organism_part', 'cell_type', 'growth_condition', 'developmental_stage', 'Material Type', 'Protocol REF', 'sample_id', 'LIBRARY_LAYOUT', 'cdna_read_size', 'cell_barcode_size', 'end_bias', 'library_construction', 'sample_barcode_size', 'umi_barcode_offset', 'umi_barcode_size', 'Performer', 'Assay Name', 'ENA_EXPERIMENT', 'ENA_RUN', 'time', 'time_unit', 'n_genes', 'doublet_scores', 'predicted_doublets', 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'percent_mito', 'n_counts_mito', 'percent_ribo', 'n_counts_ribo', 'percent_hb', 'n_counts_hb', 'percent_top50', 'cell_passed_qc', 'qc_cluster', 'cluster_passed_qc', 'consensus_fraction', 'consensus_passed_qc', 'total_counts', 'n_genes_by_counts', 'percent_chrY', 'XIST-counts', 'XIST-percentage', 'sex', 'S_score', 'G2M_score', 'Cell_cycle_phase', 'Study_name', 'ArrayExpress_ID', 'library_preparat

### Extract highly variable genes

In [134]:
adata_mesothelium.layers['counts'] = adata_mesothelium.X.copy()

In [135]:
sc.pp.highly_variable_genes(
    adata_mesothelium,
    flavor = "seurat_v3",
    n_top_genes = 1500,
    layer = "counts",
    batch_key = "library_preparation_protocol",
    subset = True,
    span = 1
)

extracting highly variable genes
--> added
    'highly_variable', boolean vector (adata.var)
    'highly_variable_rank', float vector (adata.var)
    'means', float vector (adata.var)
    'variances', float vector (adata.var)
    'variances_norm', float vector (adata.var)


## Run scVI

In [136]:
scvi.model.SCVI.setup_anndata(adata_mesothelium, 
                              categorical_covariate_keys=['sample_id', 'library_construnction_and_layout', 'Protocol REF'],
                              layer = 'counts')

In [137]:
scvi_model = scvi.model.SCVI(adata_mesothelium,
                            n_latent = 50, 
                            n_hidden = 128,
                            n_layers = 2, 
                            dropout_rate = 0.1,
                            dispersion = 'gene-batch', 
                            gene_likelihood = 'nb')

In [138]:
scvi_model.train(100, 
                early_stopping = True,
                early_stopping_patience = 10,
                check_val_every_n_epoch = 1, 
                enable_progress_bar = True)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Epoch 100/100: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:37<00:00,  1.02s/it, v_num=1, train_loss_step=515, train_loss_epoch=504]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100/100: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:37<00:00,  1.03it/s, v_num=1, train_loss_step=515, train_loss_epoch=504]


In [139]:
adata_mesothelium.obsm["X_scVI"] = scvi_model.get_latent_representation(adata_mesothelium)

#### Evaluate model performance using the [_Svensson_](https://www.nxn.se/valent/2023/8/10/training-scvi-posterior-predictive-distributions-over-epochs) method

In [140]:
history_df = (
    scvi_model.history['elbo_train'].astype(float)
    .join(scvi_model.history['elbo_validation'].astype(float))
    .reset_index()
    .melt(id_vars=['epoch'])
)

In [ ]:
plt.figure(figsize=(12, 10))

plt.plot(history_df[history_df['variable'] == 'elbo_train']['epoch'], 
         history_df[history_df['variable'] == 'elbo_train']['value'], 
         color='black', marker='o', label='Training ELBO')

plt.plot(history_df[history_df['variable'] == 'elbo_validation']['epoch'],
         history_df[history_df['variable'] == 'elbo_validation']['value'], 
         color='red', marker='o', label='Validation ELBO')

plt.xlabel('Epoch')
plt.ylabel('ELBO Value')
plt.title('Training and Validation ELBO over Epochs')
plt.legend()
plt.grid(True, alpha=0.3)

plt.xlim(left=1)

plt.tight_layout()
plt.show()

+ Visualize dataset

In [142]:
sc.pp.neighbors(adata_mesothelium, use_rep = "X_scVI", n_neighbors = 50, metric = 'minkowski')
sc.tl.umap(adata_mesothelium, min_dist = 0.3, spread = 4, random_state = 1712)

computing neighbors
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:02)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:07)


In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(10, 10))
    sc.pl.umap(adata_mesothelium,color=["Study_name", "ArrayExpress_ID", 'metadata_cluster', 'age_group', 'organism_part', 'developmental_stage', 'library_preparation_protocol', 'immunophenotype', 'sex', 'Cell_cycle_phase', 'developmental_stage', 'full_age', 'growth_condition', 'sampling_site', 'Material Type', 'donor_id', 'library_construnction_and_layout', 'Protocol REF', 'Performer'], ncols=4, frameon=False, show=False, size = 15)
    plt.savefig(f"figures/mesothelium_categorical_values1_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_mesothelium,color=['n_genes', 'n_counts', 'total_counts', 'n_genes_by_counts', 'predicted_doublets', 'percent_mito', 'percent_ribo', 'percent_hb'], color_map = 'magma_r', ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/mesothelium_continues_values1_{timestamp}.png", bbox_inches="tight")
    plt.show()

### `Leiden` clustering

In [148]:
sc.tl.leiden(adata_mesothelium, resolution = 0.4, random_state = 1786)

running Leiden clustering
    finished: found 8 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:13)


In [ ]:
sc.pl.umap(adata_mesothelium,color=['leiden', 'Integrated_05'], color_map = 'magma_r', ncols=2, frameon=False, show=False, size = 10)

## Manual Annotation

In [150]:
adata_log = adata[adata.obs['cellstates_scANVI'].isin(['Mesothelium'])]
adata_log.obsm['X_umap'] = adata_mesothelium.obsm['X_umap'].copy()
adata_log.obs['leiden'] = adata_mesothelium.obs['leiden'].copy()
sc.pp.normalize_total(adata_log, target_sum=1e6, exclude_highly_expressed=True)
sc.pp.log1p(adata_log)

normalizing counts per cell. The following highly-expressed genes are not considered during normalization factor computation:
['S100A6', 'MFSD1', 'ATG10', 'RPL30-AS1', 'CCL19', 'VIM-AS1', 'ADIRF-AS1', 'BEST1', 'TALAM1', 'ENSG00000273149', 'THBS1', 'ZFP36', 'TMSB4X', 'MT-RNR1', 'MT-RNR2', 'MT-CO1', 'MT-CO2']
    finished (0:00:00)


* Mesothelium

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'KRT19', 'LRRN4', 'UPK3B', 'PRG4', 'RGS5'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/mesothelial_mesothelial_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'WT1', 'MSLN', 'SOX6', 'IL18'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/mesothelial_earlymesothelial_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

* Mesoderm markers

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'HAND1', 'HAND2','PITX2','ZEB2', 'TBX6', 'MEOX1', 'OSR1', 'FOXF1', 'MEOX2', 'NKX2-5'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/mesothelial_mesoderm_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [154]:
adata_log.obs['cell_states'] = adata_log.obs['cell_states'].cat.add_categories(
    ['Mesoderm 2 (ZEB2+)', 
     'Mesoderm 1 (HAND1+)'])
adata_log.obs.loc[(adata_log.obs['leiden'] == '0'), 'cell_states'] = 'Mesoderm 1 (HAND1+)'
adata_log.obs.loc[(adata_log.obs['leiden'] == '2'), 'cell_states'] = 'Mesoderm 1 (HAND1+)'
adata_log.obs.loc[(adata_log.obs['leiden'] == '1'), 'cell_states'] = 'Mesoderm 2 (ZEB2+)'

* T reticular cells

In [155]:
sc.tl.score_genes(adata_log, ['CCL19', 'CCL21', 'LTBR'], score_name='T_reticular_score')

computing score 'T_reticular_score'
    finished: added
    'T_reticular_score', score of gene set (adata.obs).
    150 total control genes are used. (0:00:00)


In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'CCL19', 'CCL21', 'LTBR', 'T_reticular_score', 'cell_states'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/mesothelial_Treticular_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [157]:
adata_log.obs['cell_states'] = adata_log.obs['cell_states'].cat.add_categories(['T reticular'])

In [158]:
adata_log.obs.loc[(adata_log.obs['T_reticular_score'] > 2) & (adata_log.obs['leiden'] == '1'), 'cell_states'] = 'T reticular'

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['cellstates_scANVI', 'cell_states'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/mesothelium_manual_annotation_{timestamp}.png", bbox_inches="tight")
    plt.show()

+ Transfer annotation to full dataset

In [160]:
all_categories = pd.Categorical(
    pd.concat([adata.obs['cell_states'], adata_log.obs['cell_states']]).unique()
)

adata.obs['cell_states'] = pd.Categorical(
    adata.obs['cell_states'],
    categories=all_categories
)

adata_log.obs['cell_states'] = pd.Categorical(
    adata_log.obs['cell_states'],
    categories=all_categories
)

In [161]:
shared_indices = adata_log.obs.index
adata.obs.loc[shared_indices, 'cell_states'] = adata_log.obs['cell_states']

In [166]:
adata.obs['cell_states'] = adata.obs['cell_states'].replace('Mesothelial', 'Mesothelium')

In [167]:
del adata_mesothelium, adata_log

## Pericytes

In [168]:
adata_pericyte = adata[adata.obs['cellstates_scANVI'].isin(['Pericyte'])]
adata_pericyte

View of AnnData object with n_obs × n_vars = 15702 × 43704
    obs: 'cell_index', 'Source Name', 'ENA_SAMPLE', 'BioSD_SAMPLE', 'organism', 'disease', 'organism_part', 'cell_type', 'growth_condition', 'developmental_stage', 'Material Type', 'Protocol REF', 'sample_id', 'LIBRARY_LAYOUT', 'cdna_read_size', 'cell_barcode_size', 'end_bias', 'library_construction', 'sample_barcode_size', 'umi_barcode_offset', 'umi_barcode_size', 'Performer', 'Assay Name', 'ENA_EXPERIMENT', 'ENA_RUN', 'time', 'time_unit', 'n_genes', 'doublet_scores', 'predicted_doublets', 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'percent_mito', 'n_counts_mito', 'percent_ribo', 'n_counts_ribo', 'percent_hb', 'n_counts_hb', 'percent_top50', 'cell_passed_qc', 'qc_cluster', 'cluster_passed_qc', 'consensus_fraction', 'consensus_passed_qc', 'total_counts', 'n_genes_by_counts', 'percent_chrY', 'XIST-counts', 'XIST-percentage', 'sex', 'S_score', 'G2M_score', 'Cell_cycle_phase', 'Study_name', 'ArrayExpress_ID', 'library_preparat

### Extract highly variable genes

In [169]:
adata_pericyte.layers['counts'] = adata_pericyte.X.copy()

In [170]:
sc.pp.highly_variable_genes(
    adata_pericyte,
    flavor = "seurat_v3",
    n_top_genes = 1500,
    layer = "counts",
    batch_key = "library_preparation_protocol",
    subset = True,
    span = 1
)

extracting highly variable genes
--> added
    'highly_variable', boolean vector (adata.var)
    'highly_variable_rank', float vector (adata.var)
    'means', float vector (adata.var)
    'variances', float vector (adata.var)
    'variances_norm', float vector (adata.var)


## Run scVI

In [171]:
scvi.model.SCVI.setup_anndata(adata_pericyte, 
                              categorical_covariate_keys=['sample_id', 'library_construnction_and_layout', 'Protocol REF'],
                              layer = 'counts')

In [172]:
scvi_model = scvi.model.SCVI(adata_pericyte,
                            n_latent = 50, 
                            n_hidden = 128,
                            n_layers = 2, 
                            dropout_rate = 0.1,
                            dispersion = 'gene-batch', 
                            gene_likelihood = 'nb')

In [173]:
scvi_model.train(100, 
                early_stopping = True,
                early_stopping_patience = 10,
                check_val_every_n_epoch = 1, 
                enable_progress_bar = True)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Epoch 100/100: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [02:08<00:00,  1.20s/it, v_num=1, train_loss_step=538, train_loss_epoch=529]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100/100: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [02:08<00:00,  1.28s/it, v_num=1, train_loss_step=538, train_loss_epoch=529]


In [174]:
adata_pericyte.obsm["X_scVI"] = scvi_model.get_latent_representation(adata_pericyte)

#### Evaluate model performance using the [_Svensson_](https://www.nxn.se/valent/2023/8/10/training-scvi-posterior-predictive-distributions-over-epochs) method

In [175]:
history_df = (
    scvi_model.history['elbo_train'].astype(float)
    .join(scvi_model.history['elbo_validation'].astype(float))
    .reset_index()
    .melt(id_vars=['epoch'])
)

In [ ]:
plt.figure(figsize=(12, 10))

plt.plot(history_df[history_df['variable'] == 'elbo_train']['epoch'], 
         history_df[history_df['variable'] == 'elbo_train']['value'], 
         color='black', marker='o', label='Training ELBO')

plt.plot(history_df[history_df['variable'] == 'elbo_validation']['epoch'],
         history_df[history_df['variable'] == 'elbo_validation']['value'], 
         color='red', marker='o', label='Validation ELBO')

plt.xlabel('Epoch')
plt.ylabel('ELBO Value')
plt.title('Training and Validation ELBO over Epochs')
plt.legend()
plt.grid(True, alpha=0.3)

plt.xlim(left=1)

plt.tight_layout()
plt.show()

+ Visualize dataset

In [177]:
sc.pp.neighbors(adata_pericyte, use_rep = "X_scVI", n_neighbors = 50, metric = 'minkowski')
sc.tl.umap(adata_pericyte, min_dist = 0.3, spread = 4, random_state = 1712)

computing neighbors
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:03)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:08)


In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(10, 10))
    sc.pl.umap(adata_pericyte,color=["Study_name", "ArrayExpress_ID", 'metadata_cluster', 'age_group', 'organism_part', 'developmental_stage', 'library_preparation_protocol', 'immunophenotype', 'sex', 'Cell_cycle_phase', 'developmental_stage', 'full_age', 'growth_condition', 'sampling_site', 'Material Type', 'donor_id', 'library_construnction_and_layout', 'Protocol REF', 'Performer'], ncols=4, frameon=False, show=False, size = 15)
    plt.savefig(f"figures/pericytes_categorical_values1_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_pericyte,color=['n_genes', 'n_counts', 'total_counts', 'n_genes_by_counts', 'predicted_doublets', 'percent_mito', 'percent_ribo', 'percent_hb'], color_map = 'magma_r', ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/pericytes_continues_values1_{timestamp}.png", bbox_inches="tight")
    plt.show()

### `Leiden` clustering

In [180]:
sc.tl.leiden(adata_pericyte, resolution = 0.2, random_state = 1786)

running Leiden clustering
    finished: found 7 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:11)


In [ ]:
sc.pl.umap(adata_pericyte,color=['leiden', 'Integrated_05'], color_map = 'magma_r', ncols=2, frameon=False, show=False, size = 10)

## Manual Annotation

In [183]:
adata_log = adata[adata.obs['cellstates_scANVI'].isin(['Pericyte'])]
adata_log.obsm['X_umap'] = adata_pericyte.obsm['X_umap'].copy()
adata_log.obs['leiden'] = adata_pericyte.obs['leiden'].copy()
sc.pp.normalize_total(adata_log, target_sum=1e6, exclude_highly_expressed=True)
sc.pp.log1p(adata_log)

normalizing counts per cell. The following highly-expressed genes are not considered during normalization factor computation:
['S100A6', 'MFSD1', 'APOD', 'ENSG00000286848', 'ATG10', 'CXCL14', 'GPX3', 'COL1A2', 'CCL19', 'CCL21', 'VIM-AS1', 'ADIRF-AS1', 'ACTA2', 'TALAM1', 'MGP', 'ENSG00000273149', 'THBS1', 'COL1A1', 'ZFP36', 'FTL', 'MT-RNR1', 'MT-RNR2', 'MT-CO1']
    finished (0:00:00)


In [184]:
adata_log.obs['cell_states'] = adata_log.obs['cell_states'].cat.add_categories(
    ['Immature pericyte', 'Mesothelium', 'Stromal 3 (KCNN3+)', 'Contractile pericyte', 'Angiogenic pericyte', 'Cycling pericyte'])

* Pericytes

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'NOTCH3', 'MCAM', 'RGS5', 'KCNJ8'], color_map = 'magma_r', ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/pericytes_pericytes_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

* Undifferentiated pericyte

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'PDGFRB', 'CSPG4'], color_map = 'magma_r', ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/pericytes_undifferentiatedpericytes_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [187]:
adata_log.obs.loc[(adata_log.obs['leiden'] == '1'), 'cell_states'] = 'Immature pericyte'

* contractile pericytes

In [188]:
sc.tl.score_genes(adata_log, ['ACTA2', 'PLN', 'RERGL', 'KCNA5', 'KCNAB1', 'NRIP2'], score_name='contractile_pericyte_score')

computing score 'contractile_pericyte_score'
    finished: added
    'contractile_pericyte_score', score of gene set (adata.obs).
    299 total control genes are used. (0:00:00)


In [189]:
adata_log.obs.loc[((adata_log.obs['leiden'] == '1') & (adata_log.obs['contractile_pericyte_score'] > 2)), 'cell_states'] = 'Contractile pericyte'

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'ACTA2', 'PLN', 'RERGL', 'KCNA5', 'KCNAB1', 'NRIP2', 'contractile_pericyte_score', 'cell_states'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/pericytes_contractilepericytes_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

+ Cycling pericytes

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'MKI67', 'PCLAF'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/pericytes_cyclingpericytes_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [192]:
adata_log.obs.loc[(adata_log.obs['leiden'] == '3'), 'cell_states'] = 'Cycling pericyte'

* Angiogenic pericytes 

In [193]:
sc.tl.score_genes(adata_log, ['PRRX1','PROCR', 'ENPEP', 'ABCC8', 'COL25A1', 'TEX41'], score_name='angiogenic_pericyte_score')

computing score 'angiogenic_pericyte_score'
    finished: added
    'angiogenic_pericyte_score', score of gene set (adata.obs).
    250 total control genes are used. (0:00:00)


In [194]:
adata_log.obs.loc[((adata_log.obs['leiden'] == '1') & (adata_log.obs['angiogenic_pericyte_score'] > 1)), 'cell_states'] = 'Angiogenic pericyte'
adata_log.obs.loc[((adata_log.obs['leiden'] == '0') & (adata_log.obs['angiogenic_pericyte_score'] > 1)), 'cell_states'] = 'Angiogenic pericyte'

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'PRRX1','PROCR', 'ENPEP', 'ABCC8', 'COL25A1', 'TEX41', 'angiogenic_pericyte_score', 'cell_states'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/pericytes_contractilepericytes_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

+ Stromal 3 (KCNN3+)

In [196]:
sc.tl.score_genes(adata_log, ['C7', 'KCNN3', 'LRRC3B'], score_name='stromal3_score')

computing score 'stromal3_score'
    finished: added
    'stromal3_score', score of gene set (adata.obs).
    50 total control genes are used. (0:00:00)


In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'C3', 'CXCL13', 'CCL21'], ncols=4, frameon=False, show=False, size = 10)

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'C7', 'KCNN3', 'LRRC3B', 'stromal3_score', 'cell_states'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/pericytes_stromal3_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [198]:
adata_log.obs.loc[(adata_log.obs['leiden'] == '2'), 'cell_states'] = 'Stromal 3 (KCNN3+)'
adata_log.obs.loc[((adata_log.obs['leiden'] == '3') & (adata_log.obs['stromal3_score'] > 2)), 'cell_states'] = 'Stromal 3 (KCNN3+)'

+ Mesothelium

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata_log,color=['leiden', 'KRT19', 'LRRN4', 'UPK3B', 'PRG4', 'RGS5'], ncols=4, frameon=False, show=False, size = 10)
    plt.savefig(f"figures/pericytes_mesothelium_markers_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [200]:
adata_log.obs.loc[(adata_log.obs['leiden'] == '4'), 'cell_states'] = 'Mesothelium'

* Cluster 5

In [201]:
sc.tl.rank_genes_groups(adata_log, groupby="leiden", method="wilcoxon")

ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:18)


In [202]:
df = sc.get.rank_genes_groups_df(adata_log, group="5").head(100)
df['names'].to_list()

['VIM-AS1',
 'TALAM1',
 'ATG10',
 'ENSG00000273149',
 'RPL30-AS1',
 'ENSG00000272173',
 'ADIRF-AS1',
 'JSRP1',
 'BEST1',
 'MT-RNR2',
 'MT-ND6',
 'TAF10',
 'ENSG00000279641',
 'MT-CO1',
 'PCSK7',
 'BTG1-DT',
 'ENSG00000234961',
 'TYW1',
 'CATSPER2P1',
 'ABCB9',
 'ENO3',
 'ENSG00000260304',
 'ENSG00000273175',
 'MT-ND5',
 'SNX32',
 'ACTA2-AS1',
 'ENSG00000279198',
 'ENSG00000280160',
 'TMED2-DT',
 'SZT2',
 'ENSG00000272256',
 'DUT-AS1',
 'CNPY2-AS1',
 'ENSG00000264577',
 'ENSG00000273565',
 'ENSG00000287967',
 'ENSG00000244380',
 'ENSG00000288873',
 'EME2',
 'ENSG00000256034',
 'ENSG00000267598',
 'MT-CYB',
 'SRRM5',
 'PPP1R12A-AS1',
 'GOLGA4-AS1',
 'CTSZ',
 'SMARCA5-AS1',
 'EGILA',
 'ARHGEF39',
 'ENSG00000254400',
 'HNRNPU',
 'DCAF6',
 'ENSG00000269968',
 'ENSG00000259627',
 'ENSG00000293022',
 'ENSG00000259238',
 'ENSG00000279504',
 'ENSG00000286883',
 'ENSG00000245156',
 'TUBA1B-AS1',
 'NOP16',
 'ENSG00000267801',
 'PRADX',
 'GOLPH3-DT',
 'ENSG00000279069',
 'PGLS-DT',
 'LDLRAD2',
 'E

In [203]:
adata_log.obs.loc[(adata_log.obs['leiden'] == '5'), 'cell_states'] = 'Contractile pericyte'

* Cluster 6

In [204]:
df = sc.get.rank_genes_groups_df(adata_log, group="6").head(100)
df['names'].to_list()

['CCL21',
 'CCL19',
 'CTSH',
 'CXCL13',
 'C1S',
 'VCAM1',
 'HSPA5',
 'FDCSP',
 'NFKBIA',
 'PDLIM1',
 'RPL34',
 'RPL10P9',
 'RPS19',
 'RPL29P11',
 'SELENOM',
 'RPL13',
 'PTGDS',
 'RPS15',
 'RPL29P26',
 'RPL41',
 'CCL2',
 'RPL18A',
 'GAS5',
 'FOSB',
 'RPL32',
 'CD74',
 'RPL18',
 'MSC',
 'RPS6',
 'RPS27A',
 'ICAM1',
 'RPL37',
 'RPL39',
 'B2M-1',
 'FAU',
 'ID1',
 'TMEM176B',
 'PPP1R15A',
 'RPL11',
 'RPL10A',
 'SERTAD1',
 'MYC',
 'RPL3',
 'HAPLN3',
 'RPL15',
 'SNHG5',
 'RPS27',
 'RPL12',
 'ATF3',
 'C3',
 'RPS26',
 'LUM',
 'HSP90AA1',
 'RPL37A',
 'HERPUD1',
 'HAS2',
 'NR2F1',
 'RPL10',
 'RPL7',
 'RPL19',
 'CTSC',
 'RPLP0',
 'SERPINF1',
 'RPS15A',
 'RPL13A',
 'RPS23',
 'SLC22A3',
 'PPA1',
 'CXCL12',
 'CCDC102B',
 'PRPS2',
 'RPS23P8',
 'RACK1',
 'JUNB',
 'ICAM2',
 'PCOLCE',
 'SOD2',
 'RPS5',
 'RPS16',
 'RPL35A',
 'LGALS3',
 'RPL17P9',
 'SDF2L1',
 'EIF1',
 'GADD45B',
 'RPL23',
 'RPS12',
 'MFGE8',
 'IL6ST',
 'DNAJB1',
 'TMEM176A',
 'RPS11',
 'IL34',
 'ZFP36',
 'RPS8',
 'RPL35',
 'PRRX1',
 'LMO4'

In [210]:
adata_log.obs.loc[(adata_log.obs['leiden'] == '6'), 'cell_states'] = 'Transitional Stromal 3 (C3+)'

+ Transfer annotation to full dataset

In [211]:
all_categories = pd.Categorical(
    pd.concat([adata.obs['cell_states'], adata_log.obs['cell_states']]).unique()
)

adata.obs['cell_states'] = pd.Categorical(
    adata.obs['cell_states'],
    categories=all_categories
)

adata_log.obs['cell_states'] = pd.Categorical(
    adata_log.obs['cell_states'],
    categories=all_categories
)

In [212]:
shared_indices = adata_log.obs.index
adata.obs.loc[shared_indices, 'cell_states'] = adata_log.obs['cell_states']

In [230]:
adata.obs['cell_states'].value_counts()

cell_states
Mesoderm 2(ZEB2+)               71486
Stromal 1 (ADAMDEC1+)           24539
Myofibroblast (RSPO2+)          21183
Myofibroblast                   10317
Mesoderm 1(HAND1+)              10149
SMC (PLPP2+)                     8495
Stromal 2 (NPY+)                 8354
Pericyte                         6766
Stromal 2 (CH25H+)               4363
SMC (PART1/CAPN3+)               3395
Stromal 3 (KCNN3+)               3330
Immature pericyte                3111
Stromal 3 (C7+)                  2933
Mesothelium                      1981
Cycling pericyte                 1095
Contractile pericyte              768
Lymphoid_Stroma                   762
Cycling stromal                   706
ICC                               530
Angiogenic pericyte               438
T reticular                       213
Transitional Stromal 3 (C3+)      113
Name: count, dtype: int64

In [214]:
del adata_pericyte, adata_log

In [223]:
adata.obs['cell_states'] = adata.obs['cell_states'].replace('Mesoderm 2 (ZEB2+)', 'Mesoderm 2(ZEB2+)')
adata.obs['cell_states'] = adata.obs['cell_states'].replace('Mesoderm 1 (HAND1+)', 'Mesoderm 1(HAND1+)')

adata.obs['cell_states'] = adata.obs['cell_states'].cat.remove_unused_categories()

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(7, 7))
    sc.pl.umap(adata,color=['cell_states'], ncols=4, frameon=False, show=False, size = 3)
    plt.savefig(f"figures/final_annotation_{timestamp}.png", bbox_inches="tight")
    plt.show()

## Save prepared dataset

In [ ]:
current_history = adata.uns['processing_history'].tolist()

new_entry = json.dumps({
    'timestamp': timestamp,
    'step': 'Manually annotated cell states',
})
current_history.append(new_entry)

adata.uns['processing_history'] = current_history

In [228]:
adata.obs.rename(columns={'cell_id': 'cell_index'}, inplace=True)

In [229]:
project = 'gut'
species = 'hs'
name = 'AM'
counts = 'raw'
atribute = 'all_datasets_mesenchymal_annotated'

adata.write_h5ad(f"data/gut_data/{project}_{species}_{atribute}_{name}_{timestamp}_{counts}.h5ad")